# 🧠 StratEdge — RAG Pipeline (Groq Version)
**Stack:** Groq API (cloud LLM) + all-MiniLM-L6-v2 (local embeddings) + FAISS (vector store)

### Steps:
1. Install dependencies
2. Load & parse documents (CSV + PDFs)
3. Chunk text
4. Generate embeddings
5. Build & save FAISS index locally
6. Test retrieval + Groq generation


## Step 0 — Install Dependencies

In [1]:
!pip install faiss-cpu sentence-transformers pypdf pandas numpy groq streamlit -q

## Step 1 — Imports & Config

In [7]:
import os
import glob
import pickle
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from groq import Groq

# ── CONFIGURATION ──────────────────────────────────────────────
DOCS_FOLDER   = "./documents"           # Put your CSV + PDFs here
INDEX_PATH    = "./faiss_index/index.bin"
CHUNKS_PATH   = "./faiss_index/chunks.pkl"
EMBED_MODEL   = "all-MiniLM-L6-v2"
GROQ_API_KEY  = "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"  # Get free key from https://console.groq.com
GROQ_MODEL    = "llama-3.1-8b-instant"          # Fast + free on Groq
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 80
TOP_K         = 4
MIN_SCORE     = 0.25

os.makedirs('./faiss_index', exist_ok=True)
os.makedirs(DOCS_FOLDER, exist_ok=True)
print('✅ Config ready')

✅ Config ready


## Step 2 — Document Loaders

In [8]:
def load_pdf(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"
    return text

def load_csv(path):
    df = pd.read_csv(path)
    print(f"  CSV '{Path(path).name}': {len(df)} rows, columns: {list(df.columns)}")
    passages = []
    for _, row in df.iterrows():
        parts = [f"{col}: {val}" for col, val in row.items() if pd.notna(val)]
        passages.append(" | ".join(parts))
    return "\n".join(passages)

def load_all_documents(folder):
    docs = []
    files = glob.glob(os.path.join(folder, "**", "*"), recursive=True)
    pdf_files = [f for f in files if f.lower().endswith(".pdf")]
    csv_files = [f for f in files if f.lower().endswith(".csv")]
    print(f"Found: {len(pdf_files)} PDFs, {len(csv_files)} CSVs")
    for path in pdf_files:
        text = load_pdf(path)
        if text.strip():
            docs.append({"source": Path(path).name, "text": text})
            print(f"  ✅ PDF: {Path(path).name} ({len(text):,} chars)")
    for path in csv_files:
        text = load_csv(path)
        if text.strip():
            docs.append({"source": Path(path).name, "text": text})
            print(f"  ✅ CSV: {Path(path).name} ({len(text):,} chars)")
    return docs

docs = load_all_documents(DOCS_FOLDER)
print(f"\nTotal documents loaded: {len(docs)}")

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 23 0 (offset 0)
Ignoring wrong pointing object 25 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 35 0 (offset 0)
Ignoring wrong pointing object 46 0 (offset 0)
Ignoring wrong pointing object 48 0 (offset 0)
Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 119 0 (offset 0)
Ignoring wrong pointing object 125 0 (offset 0)
Ignoring wrong pointing object 133 0 (offset 0)
Ignoring wrong pointing object 172 0 (offset 0)
Ignoring wrong pointing object 178 0 (offset 0)
Ignoring wrong pointing object 183 0 (offset 0)


Found: 7 PDFs, 1 CSVs
  ✅ PDF: business_failure_prediction.pdf (137,312 chars)
  ✅ PDF: ftx.pdf (8,404 chars)
  ✅ PDF: hightech_startups.pdf (18,841 chars)
  ✅ PDF: peloton.pdf (10,755 chars)
  ✅ PDF: quibi.pdf (9,010 chars)
  ✅ PDF: theranos.pdf (9,160 chars)
  ✅ PDF: wework.pdf (8,959 chars)
  CSV 'startup_failures.csv': 815 rows, columns: ['Name', 'Sector', 'Years of Operation']
  ✅ CSV: startup_failures.csv (65,814 chars)

Total documents loaded: 8


## Step 3 — Chunking

In [9]:
def chunk_text(text, source, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if len(chunk) > 50:
            chunks.append({"source": source, "text": chunk, "start": start})
        start += chunk_size - overlap
    return chunks

all_chunks = []
for doc in docs:
    chunks = chunk_text(doc["text"], doc["source"])
    all_chunks.extend(chunks)
    print(f"  {doc['source']}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

  business_failure_prediction.pdf: 327 chunks
  ftx.pdf: 20 chunks
  hightech_startups.pdf: 45 chunks
  peloton.pdf: 26 chunks
  quibi.pdf: 22 chunks
  theranos.pdf: 22 chunks
  wework.pdf: 22 chunks
  startup_failures.csv: 157 chunks

Total chunks: 641


## Step 4 — Embeddings

In [10]:
print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)
print(f"✅ Model loaded: {EMBED_MODEL}")

texts = [c["text"] for c in all_chunks]
print(f"Embedding {len(texts)} chunks...")
embeddings = embedder.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)
print(f"\n✅ Embeddings shape: {embeddings.shape}")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model loaded: all-MiniLM-L6-v2
Embedding 641 chunks...


Batches:   0%|          | 0/21 [00:00<?, ?it/s]


✅ Embeddings shape: (641, 384)


## Step 5 — Build & Save FAISS Index

In [11]:
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))

faiss.write_index(index, INDEX_PATH)
with open(CHUNKS_PATH, "wb") as f:
    pickle.dump(all_chunks, f)

print(f"✅ FAISS index saved  → {INDEX_PATH}")
print(f"✅ Chunks saved       → {CHUNKS_PATH}")
print(f"   Total vectors: {index.ntotal}")

✅ FAISS index saved  → ./faiss_index/index.bin
✅ Chunks saved       → ./faiss_index/chunks.pkl
   Total vectors: 641


## Step 6 — Test with Groq API

In [12]:
def retrieve(query, index, chunks, embedder, top_k=TOP_K, min_score=MIN_SCORE):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1 and score >= min_score:
            results.append({**chunks[idx], "score": float(score)})
    return results

def build_prompt(query, context_chunks):
    context_str = ""
    for i, c in enumerate(context_chunks, 1):
        context_str += f"[Source {i}: {c['source']} | relevance: {c['score']:.2f}]\n{c['text']}\n\n"
    return f"""You are StratEdge, an expert business strategy consultant specializing in startup and company failures.
Provide evidence-based, actionable strategic advice grounded ONLY in the provided context.

STRICT RULES:
1. Answer ONLY from the context below. Do NOT use outside knowledge.
2. If context is insufficient say: I don't have enough data on this in my knowledge base.
3. Always reference the source document when citing findings.
4. Be direct, specific, and actionable.
5. Never fabricate statistics or company data.

--- CONTEXT ---
{context_str}--- END CONTEXT ---

QUESTION: {query}

STRATEGIC ANALYSIS:"""

def ask(query):
    client = Groq(api_key=GROQ_API_KEY)
    retrieved = retrieve(query, index, all_chunks, embedder)
    if not retrieved:
        return "No relevant context found.", []
    prompt = build_prompt(query, retrieved)
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=600,
        temperature=0.3,
        stream=False
    )
    return response.choices[0].message.content, [c["source"] for c in retrieved]

# Test
answer, sources = ask("Why do most startups fail?")
print("ANSWER:\n", answer)
print("\nSOURCES:", sources)

ANSWER:
 Based on the provided context, here's a strategic analysis of why most startups fail:

**Key Findings:**

1. **Lack of Strategic Advantages**: Startups fail due to a lack of strategic advantages, including resources, market, competent workforce, and location (Source 2: business_failure_prediction.pdf, relevance: 0.72).
2. **Operational Inefficiencies**: A lack of a competent workforce leads to improper management of existing resources, causing severe operational inefficiencies (Source 2: business_failure_prediction.pdf, relevance: 0.72).
3. **Resource Constraints**: High-tech startups are resource-constrained, which leads to prolonged periods of designing and perfecting products and services, resulting in failure (Source 3: business_failure_prediction.pdf, relevance: 0.71).
4. **Lack of Measuring and Monitoring Tools**: High-tech startups often lack measuring or monitoring tools, leading to unmet targets and failure (Source 4: business_failure_prediction.pdf, relevance: 0.70).

## ✅ Done!

Your FAISS index is saved in `./faiss_index/`.

### Next steps for Streamlit Cloud deployment:
1. Push your project to GitHub (include `faiss_index/` folder)
2. Go to https://share.streamlit.io
3. Connect your GitHub repo
4. Add `GROQ_API_KEY` in Streamlit Secrets
5. Deploy!

### Get your free Groq API key:
https://console.groq.com → Sign up → API Keys → Create key